- Verificar a existência de valores ausentes e decidir como irá lidar com eles.
- Construir um modelo de regressão linear considerando as variáveis que o grupo julgar necessárias (no relatório deve constar uma justificativa da escolha).
- Analisar o modelo obtido (a análise deve considerar se o modelo é significativo, se os parâmetros são significativos e o que isso significa, qual o valor do coeficiente de determinação e o que ele significa. Caso exista parâmetros não significativos, o grupo deve decidir como trata-los e apresentar o modelo final).


- O modelo final deve ser estudado quanto a normalidade dos resíduos, a multicolinearidade e a heterocedasticidade.
- Caso os resíduos não cumpra o pressuposto da normalidade, o grupo deverá realizar uma transformação de variável com a finalidade de normalizar os resíduos.
- Escreva alguma conclusão interessante que pode ser obtida do modelo encontrado.

- Realize uma Análise de Componentes Principais com os dados (não é necessário utilizar todas as variáveis).
- Decida quantos Componentes Principais são necessários.
- Interprete os Componentes Principais.

Estrutura do colab: células de código para manipulação geral (funcionará para outros datasets, contando que se mantenha o mesmo contrato de colunas) e células de markdown destinadas a análise do caso específico do dataset passado para o projeto.

In [ ]:
# Importação das bibliotecas necessárias
import pandas as pd
import statsmodels.api as sm
from statstests.process import stepwise

In [ ]:
# Definição do dataset e visualização inicial dos dados
caminho = "./dataset/Grupo 8.xlsx"
gross_data = pd.read_excel(caminho)
gross_data.head()

,Jogador,Time,Tent,Compl,Por Tent,TD,PerTD,Int,PF
0,Philip Rivers,SD,478.0,312,8.39,34,7.1,11.0,105.5
1,Chad Pennington,MIA,476.0,321,7.67,19,4.0,7.0,97.4
2,Kurt Warner,ARI,598.0,401,7.66,30,5.0,14.0,96.9
3,Drew Brees,NO,635.0,413,7.98,34,5.4,17.0,96.2
4,Peyton Manning,IND,555.0,371,7.21,27,4.9,12.0,95.0


In [ ]:
# Análise inicial do formato dos casos (número de linhas e colunas)
gross_data.shape

(32, 9)

Os dados são compostos de 32 linhas e 9 colunas, isto é, são 32 observações com 9 variáveis.
Dessas variáveis, temos:
- Jogador: nome do jogador (qualitativa nominal)
- Time: nome do time (qualitativa nominal)
- Tent: número de tentativas de passe (quantitativa discreta)
- Compl: número de passes completados (quantitativa discreta)
- Por Tent: número de jardas por tentativa (quantitativa contínua)
- TD: número de passes para touchdown (quantitativa discreta)
- PerTD: porcentagem de tentativas que são touchdown (quantitativa contínua percentual)
- Int: número de interceptações (quantitativa discreta)
- PF: número de pontos feitos (qualitativa contínua)

In [ ]:
# Verificação de valores nulos - contagem por variável
gross_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Jogador   32 non-null     str    
 1   Time      32 non-null     str    
 2   Tent      31 non-null     float64
 3   Compl     32 non-null     int64  
 4   Por Tent  32 non-null     float64
 5   TD        32 non-null     int64  
 6   PerTD     30 non-null     float64
 7   Int       31 non-null     float64
 8   PF        32 non-null     float64
dtypes: float64(5), int64(2), str(2)
memory usage: 2.4 KB


Verificou-se que há dados nulos:
- 1 valor nulo para a variável Tent
- 2 valores nulos para a variável PerTD
- 1 valor nulo para a variável Int

Há a possibilidade de que alguma linha tenha mais de um dado nulo.

In [ ]:
# Visualização das linhas com valores nulos
gross_data[gross_data.isna().any(axis=1)]

,Jogador,Time,Tent,Compl,Por Tent,TD,PerTD,Int,PF
23,Ben Roethlisberger,PIT,469.0,281,7.04,17,NaN,NaN,80.1
24,Kyle Orton,CHI,466.0,281,6.70,18,NaN,15.0,80.1
25,JaMarcus Russell,OAK,NaN,198,6.39,13,3.5,8.0,79.6


De fato, uma das linhas (ID 23) apresenta dois dados nulos.
Dada a baixa incidência de linhas com dados nulos (menos que 10%) e a relevância (discutida abaixo) das colunas correspondentes para a discussão do problema e determinação de possíveis modelos, optou-se por excluí-las.

In [ ]:
# Exclusão de valores nulos
data=gross_data.dropna()

In [ ]:
# Análise do formato após remoção de nulos
data.shape

(29, 9)

Removidas as 3 linhas com dados nulos, agora sobram 29.

In [ ]:
# Verificação de nulos após remoção
data.info()

<class 'pandas.DataFrame'>
Index: 29 entries, 0 to 31
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Jogador   29 non-null     str    
 1   Time      29 non-null     str    
 2   Tent      29 non-null     float64
 3   Compl     29 non-null     int64  
 4   Por Tent  29 non-null     float64
 5   TD        29 non-null     int64  
 6   PerTD     29 non-null     float64
 7   Int       29 non-null     float64
 8   PF        29 non-null     float64
dtypes: float64(5), int64(2), str(2)
memory usage: 2.3 KB


In [10]:
# Quantidade de valores distintos por coluna
data.nunique()

Jogador     29
Time        29
Tent        28
Compl       27
Por Tent    28
TD          19
PerTD       22
Int         13
PF          28
dtype: int64

A escolha inicial mantém `PF` como variável dependente por ser a métrica-resumo de desempenho do passador (exatamente o que se busca explicar) e adota como independentes as seis variáveis numéricas restantes: `Tent`, `Compl`, `Por Tent`, `TD`, `PerTD` e `Int`.

As colunas `Jogador` e `Time` foram excluídas:
- `Jogador` é identificador único (uma observação por nome) e não carrega informação para a modelagem;
- `Time` até poderia ter algum valor em comum entre diferentes linhas, mas, como observado acima (29 valores únicos dentre as 29 linhas), há também uma observação por time. Sendo assim, também não carrega informação para a modelagem.

Justificativa de cada independente:

- `Tent`: volume de oportunidades de pontuar - mais tentativas implicam, em princípio, mais chances de gerar pontos;
- `Compl`: passes que efetivamente avançaram a jogada - mede a efetividade absoluta do passador;
- `Por Tent`: eficiência de avanço por jogada, independente do volume;
- `TD`: conversões diretas em pontos - é o evento que mais contribui para `PF`;
- `PerTD`: taxa de conversão de tentativas em touchdown - mede eficiência de pontuação relativa ao volume.
- `Int`: variável "punitiva" - espera-se coeficiente negativo, pois interceptações resultam em perda de posse.

Algumas dessas variáveis devem apresentar associação forte entre si:
- `Compl` é, por definição, limitado por `Tent` (não se pode completar mais passes do que se tentou) e jogadores com mais tentativas tendem a acumular mais completos;
- `TD` também tende a crescer com o volume de tentativas;
- já `Por Tent` e `PerTD` capturam, sob ângulos diferentes, a dimensão de eficiência por jogada, e podem se sobrepor em parte;
- `PerTD` compartilha com `TD` a informação sobre conversões em touchdown, ainda que cada uma traga conteúdo próprio (volume absoluto versus taxa);
É plausível, portanto, que algumas variáveis apresentem grande colinearidade e parte dos coeficientes não seja significativa, sendo descartados no modelo final.

In [ ]:
# Estimação do modelo de regressão linear múltipla com as variáveis selecionadas
modelo_completo = sm.OLS.from_formula(
    'PF ~ Tent + Compl + Q("Por Tent") + TD + PerTD + Int',
    data
).fit()

# Parâmetros do 'modelo_completo'
modelo_completo.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                     PF   R-squared:                       0.936
Model:                            OLS   Adj. R-squared:                  0.919
Method:                 Least Squares   F-statistic:                     53.67
Date:                Mon, 25 May 2026   Prob (F-statistic):           5.06e-12
Time:                        09:30:54   Log-Likelihood:                -65.057
No. Observations:                  29   AIC:                             144.1
Df Residuals:                      22   BIC:                             153.7
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept        25.8248     12.469      2.071      0.050      -0.035      51.684
Tent             -0.0593      0.038     -1.571      0.130      -0.137       0.019
Compl             0.2005      0.043      4.626      0.000       0.111       0.290
Q("Por Tent")     3.8654      0.874      4.423      0.000       2.053       5.678
TD               -1.0170      0.678     -1.501      0.148      -2.422       0.388
PerTD             8.2937      3.342      2.482      0.021       1.364      15.224
Int              -1.0024      0.164     -6.122      0.000      -1.342      -0.663
==============================================================================
Omnibus:                        1.925   Durbin-Watson:                   2.323
Prob(Omnibus):                  0.382   Jarque-Bera (JB):                0.803
Skew:                           0.298   Prob(JB):                        0.669
Kurtosis:                       3.557   Cond. No.                     1.42e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.42e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

**Análise do modelo completo**

Significância global (teste F):
- F = 53,67 com p-valor = 5,06·10⁻¹² ≪ 0,05.
- Rejeita-se a hipótese nula de que todos os coeficientes (exceto o intercepto) sejam simultaneamente iguais a zero. Dessa forma, pode-se afirmar que, sob um grau de confiança de 95% de certeza, o modelo como um todo é estatisticamente significativo: o conjunto das seis variáveis explica `PF` muito além do que se esperaria pelo acaso (ou pela média).

Coeficiente de determinação:
- R² = 0,936. 93,6% da variabilidade dos pontos feitos (variável dependente) é explicada pela combinação linear das variáveis dependentes no modelo em questão.
- R² ajustado = 0,919. Penalizando o número de parâmetros, o ajuste continua alto, indicando que o ganho de R² não é apenas inflado pela quantidade de variáveis incluídas.

Significância individual dos parâmetros (testes t, α = 0,05):

- Parâmetros significativos (contribuições para explicar `PF` são distinguíveis de zero num grau de confiança de 95% de certeza, mesmo após controlar pelas demais):
    - `Compl` (p < 0,001);
    - `Por Tent` (p < 0,001);
    - `Int` (p < 0,001);
    - `PerTD` (p = 0,021).

- Parâmetros não significativos (contribuições para explicar `PF` no modelo não são distinguíveis de zero/ruído no mesmo grau de confiança. Padrão consistente com a hipótese de colinearidade levantada):
    - `TD` (p = 0,148);
    - `Tent` (p = 0,130).

Tratamento das variáveis não significativas:
- Aplicar o procedimento backward (stepwise), removendo iterativamente a variável de maior p-valor acima do limiar, reestimando o modelo a cada passo. Essa abordagem iterativa é importante porque, em presença de colinearidade, retirar uma variável pode alterar a significância das que permanecem.

In [13]:
# Eliminação backward das variáveis não significativas (limiar de p-valor = 0,05)
modelo_final = stepwise(modelo_completo, pvalue_limit=0.05)

# Parâmetros do modelo após o procedimento stepwise
modelo_final.summary()

Regression type: OLS 

Estimating model...: 
 PF ~ Q('Tent') + Q('Compl') + Q('Q("Por Tent")') + Q('TD') + Q('PerTD') + Q('Int')

 Discarding atribute "Q('TD')" with p-value equal to 0.14767051034639722 

Estimating model...: 
 PF ~ Q('Tent') + Q('Compl') + Q('Q("Por Tent")') + Q('PerTD') + Q('Int')

 No more atributes with p-value higher than 0.05

 Atributes discarded on the process...: 

{'atribute': "Q('TD')", 'p-value': np.float64(0.14767051034639722)}

 Model after stepwise process...: 
 PF ~ Q('Tent') + Q('Compl') + Q('Q("Por Tent")') + Q('PerTD') + Q('Int') 

                            OLS Regression Results                            
Dep. Variable:                     PF   R-squared:                       0.930
Model:                            OLS   Adj. R-squared:                  0.914
Method:                 Least Squares   F-statistic:                     60.65
Date:                Mon, 25 May 2026   Prob (F-statistic):           1.76e-12
Time:                        09

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                     PF   R-squared:                       0.930
Model:                            OLS   Adj. R-squared:                  0.914
Method:                 Least Squares   F-statistic:                     60.65
Date:                Mon, 25 May 2026   Prob (F-statistic):           1.76e-12
Time:                        09:47:40   Log-Likelihood:                -66.470
No. Observations:                  29   AIC:                             144.9
Df Residuals:                      23   BIC:                             153.1
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
======================================================================================
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             42.5540      5.735      7.420      0.000      30.689      54.419
Q('Tent')             -0.0938      0.031     -3.057      0.006      -0.157      -0.030
Q('Compl')             0.1899      0.044      4.325      0.000       0.099       0.281
Q('Q("Por Tent")')     4.2887      0.849      5.049      0.000       2.532       6.046
Q('PerTD')             3.3672      0.640      5.259      0.000       2.043       4.692
Q('Int')              -1.0082      0.168     -5.998      0.000      -1.356      -0.661
==============================================================================
Omnibus:                        7.105   Durbin-Watson:                   1.942
Prob(Omnibus):                  0.029   Jarque-Bera (JB):                5.612
Skew:                           0.754   Prob(JB):                       0.0605
Kurtosis:                       4.540   Cond. No.                     6.24e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 6.24e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

**Análise do modelo final**

O procedimento stepwise descartou apenas `TD` (p = 0,148, o maior do modelo completo). Após essa remoção, todas as demais variáveis permaneceram significativas a 5%, e o procedimento parou.

Observação importante sobre a colinearidade: no modelo completo, `Tent` tinha p = 0,130 (não significativo). Após remover `TD`, o p-valor de `Tent` caiu para 0,006 - agora altamente significativo. Esse é um exemplo direto do efeito da multicolinearidade: a inclusão de `TD` mascarava a contribuição de `Tent`, e basta retirar uma das colineares para o efeito da outra reaparecer.

Modelo final - `PF ~ Tent + Compl + Por Tent + PerTD + Int`:
- F = 60,65 com p-valor = 1,76·10⁻¹² ≪ 0,05 - modelo significativo, com p-valor ainda menor que o do modelo completo.
- R² = 0,930 (queda de 0,006 em relação ao modelo completo) e R² ajustado = 0,914 (queda de apenas 0,005). A perda de poder explicativo ao descartar `TD` é desprezível, o que confirma que essa variável não trazia informação adicional relevante.
- Todos os cinco coeficientes têm p-valor ≤ 0,006.

Interpretação dos coeficientes:
- `Compl` (+0,190): cada passe completo adicional eleva `PF` em 0,19 ponto — efeito do volume de avanço (conquista de campo pelo passe completo).
- `Por Tent` (+4,289): cada jarda adicional por tentativa eleva `PF` em 4,29 pontos — a eficiência de avanço por jogada tem o maior efeito marginal entre as variáveis.
- `PerTD` (+3,367): cada ponto percentual a mais na taxa de conversão em touchdown eleva `PF` em 3,37 pontos.
- `Int` (−1,008): cada interceptação a mais reduz `PF` em 1,01 ponto — sinal negativo conforme antecipado, ratificando a interpretação de "variável punitiva".
- `Tent` (−0,094): após controlar pelas demais, cada tentativa adicional reduz `PF` em 0,09 ponto. À primeira vista contraintuitivo, mas coerente: dado o volume de passes completos (`Compl`), a eficiência (`Por Tent`, `PerTD`) e o erro (`Int`), aumentar a contagem bruta de tentativas significa ter mais tentativas que não se converteram nem em completos nem em touchdowns, ou seja, o desempenho por tentativa é pior.

Equação do modelo final:

`PF = 42,554 − 0,094·Tent + 0,190·Compl + 4,289·Por Tent + 3,367·PerTD − 1,008·Int`